# Method 작성과 실행

Decision Layer는 semantic layer에 정의된 지표를 등록된 Method로 분석합니다. AI가 이 Method를 실행하고, 입력과 결과를 Run에 기록합니다.

이 노트북에서는 **함수 작성 → 로컬 등록 → 입력 지정 → 실행 결과 확인**을 직접 해봅니다.

1. 그룹별 조회를 만들고 수정합니다.
2. 여러 조회를 하나의 Method로 조합합니다.
3. DoWhy 계산을 연결하고 이진 처치와 연속형 처치를 분석합니다.
4. 같은 함수를 실제 Cube 데이터로 실행합니다.

먼저 준비된 작은 예제 데이터를 사용하므로 서버나 AI 연결 없이 시작할 수 있습니다.


## 1. 준비

저장소의 `.venv` 커널을 선택하고 아래 셀부터 실행합니다. 이 셀은 예제와 결과 표시에 필요한 코드를 준비합니다.


In [9]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

repository = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/decision_layer").is_dir()
)
sys.path.insert(0, str(repository))

from decision_layer.dev import MethodSession
from decision_layer.methods import (
    Dimension, Entity, Estimate, Filter, Measure, Refused, Scope, Table, method,
)
import examples.methods.statistics.teaching_data as example_data


def show_table(data):
    if isinstance(data, dict):
        data = [{"field": key, "value": value} for key, value in data.items()]
    display(pd.DataFrame(data))

## 일반 Python 함수부터 등록하기

계산 함수에 `async`, `ctx`, `Measure`를 붙일 필요가 없는 경로입니다. [native_calculation.py](../../../examples/methods/authoring/native_calculation.py)의 `rank_values(frame, descending=True)`는 pandas만 사용하는 일반 함수입니다. 같은 값에는 같은 순위를 줍니다.

먼저 직접 호출하고, 다음 셀에서 **그 함수를 바꾸지 않고** 입력 연결을 등록합니다. 여기의 작은 메모리 표는 계산 단위 시험이며, 제품에 로컬 CSV·파일 입력을 추가하는 예제가 아닙니다.


In [ ]:
from examples.methods.authoring.native_calculation import rank_values

original_frame = pd.DataFrame({"value": [30, 20, 20], "group": ["A", "B", "C"]})
standalone_ranking = rank_values(original_frame)
assert standalone_ranking["rank"].tolist() == [1, 2, 2]
show_table(standalone_ranking.to_dict(orient="records"))


### 컬럼 연결과 실행 계약 등록

`value → metric`, `group → group`은 함수의 컬럼 이름과 실행 시 지정할 입력 역할의 연결입니다. `metric`에는 발견한 지표 참조를 넣습니다. 매출의 SQL 정의를 여기에 쓰지 않습니다.

이 경로는 하나의 집계 표 또는 개체별 표, 이름·버전, `Table`/`Estimate` 출력, 타입이 있는 단순 옵션을 지원합니다. 기본 검사는 행 수·필수 값·유한한 지표·중복 grain을 확인합니다. 추가 실행 검사는 `validators=[...]`, 복잡한 반환값 변환은 `adapt=...`로 등록합니다. 소스 완전성·공식 지표 일치·인과 가정은 자동 검증하지 않습니다.


In [ ]:
from decision_layer.methods import InputTable, calculation

native_rank = calculation(
    rank_values,
    name="workshop.native_rank",
    version="1.0.0",
    table=InputTable(
        measures={"value": "metric"},
        dimensions={"group": "group"},
        format="dataframe",
    ),
    requires_period=True,
)


### 같은 계산을 실행기로 호출하고 근거 확인

이 셀은 정확한 조회 요청과 준비된 응답을 대조하는 fixture 시험입니다. Cube 네트워크 통합 시험이 아닙니다. `CalculationMethod`도 기존 Registry·조회 한도·Run 계약을 사용합니다. 아래의 기존 `@method` 경로는 여러 조회·기간·필터를 직접 조합할 때 사용합니다.

노트북 함수는 신뢰하는 로컬 시험에서 사용할 수 있습니다. 서버 계산 worker에서는 import 가능한 Python 모듈의 함수를 등록해야 합니다. worker의 시간 제한과 환경변수 제거는 임의 Python 코드에 대한 파일·네트워크 sandbox가 아닙니다.


In [ ]:
from examples.methods.authoring.native_calculation import fixture_session
from examples.methods.first_method.fixtures import METRIC, DIMENSION, DATE, DATES

native_session = await fixture_session()
native_session.register(native_rank, replace=True)
native_trial = await native_session.run(
    native_rank, metric=METRIC, group=DIMENSION, scope=Scope(DATES, DATE),
)
assert native_trial.result.primary.data == standalone_ranking.to_dict(orient="records")
assert native_trial.result.status == "success"
show_table(native_trial.result.primary.data)
show_table([item.model_dump() for item in native_trial.result.validation])
assert len(native_trial.queries) == 1
native_session.close()


## 2. 그룹별 조회 Method 만들기

지표를 하나 이상의 기준으로 나누고, 첫 번째 지표 값이 큰 순서로 상위 `limit`개를 가져옵니다. 하나의 지표와 분류 기준도 목록에 넣어 사용합니다.


In [ ]:
@method("workshop.by_group", version="1.0.0")
async def my_by_group(ctx, *, selected_measures: list[Measure],
                      group_by: list[Dimension], limit: int = 3) -> Table:
    """지표를 그룹별로 조회하고 첫 번째 지표 값이 큰 순서대로 반환합니다."""
    data = await ctx.aggregate(
        measures={f"value_{i}": ref for i, ref in enumerate(selected_measures, 1)},
        dimensions={f"group_{i}": ref for i, ref in enumerate(group_by, 1)},
        order=[(selected_measures[0], "desc")],
        limit=limit,
    )
    return Table.from_dataset(data, require_non_empty=True)

`@method(...)`에 호출할 이름과 버전을 적습니다. `workshop.by_group`은 실습용 이름입니다.

함수 바로 아래의 `"""..."""`는 docstring입니다. 이 문장이 Method 설명으로 등록되어 AI에 전달됩니다.


`ctx`는 함수가 받는 데이터 조회 객체입니다. `ctx.aggregate(...)`에 조회 내용을 지정합니다.

- `dimensions`: 그룹을 나눌 기준과 결과 열 이름. 여기서는 `group_1`, `group_2`입니다.
- `measures`: 조회할 지표와 결과 열 이름. 여기서는 `value_1`, `value_2`입니다.
- `order`, `limit`: 첫 번째 지표로 내림차순 정렬하고 가져올 행 수를 제한합니다.

`Table.from_dataset(data)`는 조회 결과를 표로 반환합니다. `require_non_empty=True`는 결과가 비었을 때 실행을 거절합니다.


### 등록하고 실행하기

서버에 올리기 전에 노트북에서 방금 작성한 함수를 시험합니다.

- `example_session("by_group")`: 이 예제의 데이터를 조회할 로컬 실행 환경을 만듭니다.
- `register(my_by_group)`: 작성한 함수를 이 환경에 등록합니다.
- `run(...)`: 지표, 분류 기준, 기간을 넣고 실행합니다.

`cube://practice/observations/value`는 연습 연결 `practice`의 `observations` 모델에 있는 `value` 지표를 가리킵니다. 실제 데이터에서는 연결한 semantic layer의 경로를 사용합니다.


In [201]:
group_session = await example_data.example_session("by_group")
group_session.register(my_by_group)

trial = await group_session.run(
    my_by_group,
    selected_measures=["cube://practice/observations/value"],
    group_by=["cube://practice/observations/group"],
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
    limit=3,
)

show_table(trial.result.primary.data)

group_session.close()

,group_1,value_1
0,A,30
1,B,20
2,C,10


`run(...)`은 연결과 `scope`의 기간, 필터를 담은 `ctx`를 만들어 함수에 전달합니다. `selected_measures`, `group_by`, `limit`은 함수의 같은 이름 입력으로 전달됩니다.

이 예제의 조회 결과는 `30, 20, 10`입니다. 실행 셀을 다시 실행하면 새 연습 환경에서 같은 결과를 확인할 수 있습니다.


### 등록된 설명 확인

아래 셀은 AI가 확인할 Method 이름, docstring 설명, 필수 입력을 로컬 환경에서 보여줍니다.


In [ ]:
method_choices = group_session.find_methods()
show_table([
    {"method": item["manifest"]["name"],
     "description": item["manifest"]["description"],
     "inputs": ", ".join(item["required_inputs"])}
    for item in method_choices["candidates"]
])


### AI 요청에서 Method 실행까지

예: “2001년 2월 store 364의 판매액을 department별 상위 3개 보여줘.”

서버에 `workshop.by_group`이 등록되어 있다면 AI는 다음 MCP 도구로 정보를 확인하고 실행합니다.

| MCP 도구 | 가져오는 정보 또는 하는 일 | 활용 |
| --- | --- | --- |
| `search_semantic` | 지표, 분류 기준, 날짜, 필터에 사용할 객체의 이름과 경로 | 판매액, department, 거래 날짜, store를 찾음 |
| `find_methods` | 후보 Method의 설명과 입력 조건 | 질문에 맞는 후보를 좁힘 |
| `find_recipes` | 재사용할 분석 절차와 지원하는 목표 | 맞는 절차가 있으면 사용하고, 없으면 Method를 직접 선택 |
| `describe_method` | 선택한 Method의 `description`, `roles`, `parameters` | docstring, 지표와 분류 입력, `limit` 기본값을 확인 |
| `start_analysis` | 원래 질문, 기간, 공통 필터로 Run 생성 | 2001년 2월과 store 364 조건을 기록. Recipe를 쓰면 `start_run` 사용 |
| `run_step` | Method 이름과 입력으로 실행 | 아래 입력을 `workshop.by_group`에 전달 |
| `complete_run` | 결과에 근거한 답변과 실행 기록 저장 | 사용자에게 결과와 Run 링크 제공 |

`describe_method("workshop.by_group")`의 입력 정보는 위 함수에서 생성됩니다.

| 함수 입력 | `describe_method`에서 확인한 정보 | 요청에 맞춰 넣을 값 |
| --- | --- | --- |
| `selected_measures` | 지표 경로 목록, 필수 | `["cube://journey/transaction/receipts"]` |
| `group_by` | 분류 기준 경로 목록, 필수 | `["cube://journey/product/department"]` |
| `limit` | 정수, 기본값 `3` | `3` |

이 경로는 `search_semantic`에서 확인한 semantic 모델의 항목입니다. 연습 셀에서는 `journey` 대신 준비된 `practice` 모델의 경로를 넣습니다.

AI는 확인한 경로를 `run_step`의 `measures={"selected_measures": [...]}`, `dimensions={"group_by": [...]}`에 넣고 `params={"limit": 3}`을 전달합니다. 기간과 store 필터는 Run의 공통 조건으로 적용됩니다. 불명확한 지표나 기간, 여러 가능한 분석이 있으면 사용자에게 확인합니다.

이번 노트북에서는 AI 대신 `run(...)`에 같은 입력을 직접 넣습니다. 노트북의 `register(...)`는 로컬 환경에만 등록하며, 웹이나 MCP에서 사용하려면 Python 파일의 함수를 서버 시작 코드에서 등록해야 합니다.


### 조회 기록 확인

결과가 예상과 다르면 먼저 실제 요청의 기간, 필터, 실행 상태를 확인합니다.


In [202]:
show_table([
    {
        "query": index,
        "label": attempt.label,
        "status": attempt.status,
        "period": attempt.spec.time.date_range,
        "filters": [item.model_dump() for item in attempt.spec.filters],
    }
    for index, attempt in enumerate(trial.attempts, start=1)
])

,query,label,status,period,filters
0,1,None,success,"(2026-02-01, 2026-02-28)",[]


### 지표와 분류 기준 여러 개 넣기

같은 Method에 지표 두 개와 분류 기준 두 개를 넣습니다. `group_1`과 `group_2`의 조합마다 두 지표를 조회하고, 첫 번째 지표로 정렬합니다.

In [203]:
multi_session = await example_data.example_session("multi_group")
multi_session.register(my_by_group)

multi_trial = await multi_session.run(
    my_by_group,
    selected_measures=[
        "cube://practice/observations/value",
        "cube://practice/observations/before_value",
    ],
    group_by=[
        "cube://practice/observations/group",
        "cube://practice/observations/subject",
    ],
    scope=Scope(("2026-02-01", "2026-02-28"), "cube://practice/observations/date"),
    limit=3,
)

assert multi_trial.result.status == "success"
assert multi_trial.result.primary.data[0] == {
    "value_1": 30, "value_2": 25, "group_1": "A", "group_2": "X",
}
show_table(multi_trial.result.primary.data)
multi_session.close()

,group_1,group_2,value_1,value_2
0,A,X,30,25
1,B,Y,20,15
2,C,Z,10,5


### 월별 조회 만들기

`time_granularity="month"`로 월별 값을 조회합니다. 결과는 그룹별 조회와 같은 `Table`이며 첫 열에 날짜가 표시됩니다.


In [204]:
@method("workshop.monthly", version="1.0.0")
async def my_monthly(ctx, *, selected_measures: list[Measure]) -> Table:
    """지정한 기간의 지표를 월별로 조회합니다. 변화의 통계적 유의성은 검정하지 않습니다."""
    data = await ctx.aggregate(
        measures={f"value_{i}": ref for i, ref in enumerate(selected_measures, 1)},
        time_granularity="month",
    )
    return Table.from_dataset(data, require_non_empty=True)

In [205]:
monthly_session = await example_data.example_session("trend")
monthly_session.register(my_monthly)

monthly_trial = await monthly_session.run(
    my_monthly,
    selected_measures=["cube://practice/observations/value"],
    scope=Scope(
        ("2026-01-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
)

show_table(monthly_trial.result.primary.data)
monthly_session.close()

,time,value_1
0,2026-01-01,20
1,2026-02-01,30


## 3. 코드 수정: 순위 추가

그룹별 조회 결과에 `rank`를 추가합니다. 같은 이름에 새 버전 `1.1.0`을 붙여 수정한 함수를 구분합니다.


In [206]:
@method("workshop.by_group", version="1.1.0")
async def my_ranked_by_group(ctx, *, selected_measures: list[Measure],
                             group_by: list[Dimension], limit: int = 3) -> Table:
    """첫 번째 지표 값이 큰 순서로 정렬하고 그룹별 순위를 추가합니다."""
    data = await ctx.aggregate(
        measures={f"value_{i}": ref for i, ref in enumerate(selected_measures, 1)},
        dimensions={f"group_{i}": ref for i, ref in enumerate(group_by, 1)},
        order=[(selected_measures[0], "desc")],
        limit=limit,
    )
    rows = data.records()
    for rank, row in enumerate(rows, start=1):
        row["rank"] = rank
    return Table(rows)

### 수정한 함수 실행

새 연습용 session에 수정한 Method만 등록해 실행합니다. 결과에 순위 `1, 2, 3`이 추가됩니다.

In [207]:
ranked_session = await example_data.example_session("by_group")
ranked_session.register(my_ranked_by_group)

edited_trial = await ranked_session.run(
    my_ranked_by_group,
    selected_measures=["cube://practice/observations/value"],
    group_by=["cube://practice/observations/group"],
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
    limit=3,
)

assert edited_trial.result.status == "success"
assert [row["rank"] for row in edited_trial.result.primary.data] == [1, 2, 3]
assert [row["value_1"] for row in edited_trial.result.primary.data] == [30, 20, 10]
show_table(edited_trial.result.primary.data)

ranked_session.close()

,group_1,value_1,rank
0,A,30,1
1,B,20,2
2,C,10,3


## 4. 여러 조회를 하나의 Method로 조합하기

비교에 필요한 데이터를 같은 함수에서 세 번 조회합니다. 각 응답에서 값을 꺼내 차이를 계산하고 하나의 표로 반환합니다.

| 조회 | 조건 |
| --- | --- |
| `subject` | 그룹 X에 속한 대상 A |
| `peers` | 그룹 X에서 A를 제외 |
| `overall` | 전체에서 A를 제외 |

각 조회에 같은 `ctx`를 사용하므로 기간과 공통 필터가 함께 적용됩니다. 공통 필터에 대상 A를 넣으면 비교 집단까지 A로 제한되므로 함수에서 거절합니다.


In [208]:
@method(
    "workshop.peer",
    version="1.0.0",
)
async def my_peer(
    ctx,
    *,
    measure: Measure,
    subject: Dimension,
    category: Dimension,
    subject_value: str,
    category_value: str,
) -> Table:
    """대상을 명시적으로 선택한 비교 집단과 비교합니다. 값의 차이만으로 인과관계를 판단하지 않습니다."""
    if any(item.member == subject for item in ctx.scope.filters):
        raise Refused("Keep the subject out of shared filters")

    same_category = Filter(
        member=category,
        operator="equals",
        values=[category_value],
    )
    selected = Filter(
        member=subject, 
        operator="equals", 
        values=[subject_value]
    )
    excluded = Filter(member=subject, operator="notEquals", values=[subject_value])

    subject_data = await ctx.aggregate(
        measures={"value": measure},
        filters=[same_category, selected],
        label="subject",
    )
    peer_data = await ctx.aggregate(
        measures={"value": measure},
        filters=[same_category, excluded],
        label="peers",
    )
    overall_data = await ctx.aggregate(
        measures={"value": measure},
        filters=[excluded],
        label="overall",
    )

    subject_result = subject_data.scalar("value")
    peer_result = peer_data.scalar("value")
    overall_result = overall_data.scalar("value")

    return Table(
        [
            {
                "comparison": "peers",
                "subject": subject_result,
                "benchmark": peer_result,
                "difference": subject_result - peer_result,
            },
            {
                "comparison": "overall",
                "subject": subject_result,
                "benchmark": overall_result,
                "difference": subject_result - overall_result,
            },
        ],
        warnings=["Descriptive comparison; no causal claim or significance test."],
    )

### 등록하고 실행

대상 A와 그룹 X를 지정합니다. 준비된 응답은 `12, 8, 6`이므로 차이는 `4, 6`입니다.

In [209]:
peer_session = await example_data.example_session("peer")
peer_session.register(my_peer)

peer_trial = await peer_session.run(
    my_peer,
    measure="cube://practice/observations/rate",
    subject="cube://practice/observations/subject",
    category="cube://practice/observations/group",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
    subject_value="A",
    category_value="X",
)

assert peer_trial.result.status == "success"
assert [row["difference"] for row in peer_trial.result.primary.data] == [4, 6]
assert len(peer_trial.attempts) == 3
show_table(peer_trial.result.primary.data)

peer_session.close()

,comparison,subject,benchmark,difference
0,peers,12,8,4
1,overall,12,6,6


### 세 조회 확인

각 조회의 label과 필터를 비교합니다. 직접 수정해보려면 결과에 `ratio`를 추가해보세요.

In [210]:
show_table([
    {
        "query": index,
        "label": attempt.label,
        "status": attempt.status,
        "filters": [item.model_dump() for item in attempt.spec.filters],
    }
    for index, attempt in enumerate(peer_trial.attempts, start=1)
])

,query,label,status,filters
0,1,subject,success,[{'member': 'cube://practice/observations/grou...
1,2,peers,success,[{'member': 'cube://practice/observations/grou...
2,3,overall,success,[{'member': 'cube://practice/observations/subj...


## 5. Python 계산을 Method에 연결하기

DoWhy 회귀 계산에 필요한 개체별 데이터를 조회하고, 계산 결과를 반환하는 Method를 만듭니다.

| 입력 | 이 예제에서 넣을 값 |
| --- | --- |
| `outcome` | 결과 지표 |
| `treatment` | 0 또는 1인 처치 |
| `confounders` | 처치 전에 측정한 조정 변수 |
| `unit` | 고객처럼 관측 하나를 나타내는 개체 |

이 작성 예제는 이진 처치입니다. 뒤에서는 기본 제공 Method로 연속형 처치도 실행합니다. 분석 가정은 미검증으로 남기며 신뢰구간은 제공하지 않습니다.


### 계산 코드 준비

일반 Python 파일 [binary_regression.py](../../../examples/methods/statistics/binary_regression.py)의 계산 함수를 가져옵니다. 파일에서 결측, 중복, 표본과 관측 지원을 검사하고 DoWhy의 `model.estimate_effect(..., method_name="backdoor.linear_regression")`를 직접 호출합니다.

계산 파일을 열어 구현을 검토하거나 수정할 수 있습니다. 여기서는 같은 함수를 데이터 조회에 연결합니다.


In [ ]:
from examples.methods.statistics.binary_regression import estimate_binary_regression


### 조회와 계산 연결

- `ctx.entities(...)`: 선택한 지표와 변수를 개체별로 조회합니다.
- `ctx.compute(...)`: 조회한 DataFrame을 계산 함수에 전달합니다.
- `Estimate(...)`: 추정값과 분석 상세 정보를 반환합니다.


In [212]:
@method(
    "workshop.binary_regression",
    version="1.0.0",
    execution="dataframe",
    requires_packages={
        "dowhy": "==0.13",
        "numpy": ">=1.26,<2",
        "pandas": ">=2.2,<3",
        "scikit-learn": ">=1.2,<1.7",
    },
)
async def my_binary_regression(
    ctx,
    *,
    outcome: Measure,
    treatment: Dimension,
    confounders: list[Dimension | Measure],
    unit: Entity,
) -> Estimate:
    # Extract governed entity data, then run ordinary Python calculation code.
    observations = await ctx.entities(
        entity=unit,
        measures={"outcome": outcome},
        dimensions={
            "treatment": treatment,
            "confounders": confounders,
        },
    )
    payload = await ctx.compute(
        estimate_binary_regression,
        observations.frame,
        observations.columns("confounders"),
    )

    return Estimate(
        payload["value"],
        details={
            **payload,
            "assumptions_status": "not_verified",
        },
        warnings=[
            "Unverified assumptions and placebo diagnostics do not establish "
            "causality; no interval is reported."
        ],
    )

### 등록하고 실행

결과, 처치, 조정 변수, 분석 단위를 연결합니다. 예제 데이터의 생성식은 `Y = 2*T + 3*W + noise`이며 기대값은 약 `2`입니다.

In [213]:
regression_session = await example_data.example_session("dowhy")
regression_session.register(my_binary_regression)

dowhy_trial = await regression_session.run(
    my_binary_regression,
    outcome="cube://practice/observations/value",
    treatment="cube://practice/observations/treated",
    confounders=["cube://practice/observations/baseline"],
    unit="cube://practice/observations/id",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
)

assert dowhy_trial.result.status == "success"
assert abs(dowhy_trial.result.primary.data["value"] - 2) < 0.15
show_table(dowhy_trial.result.primary.data)

regression_session.close()


/Users/hc.cho/Workspace/decision-layer/.venv/lib/python3.11/site-packages/dowhy/causal_estimators/regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
/Users/hc.cho/Workspace/decision-layer/.venv/lib/python3.11/site-packages/dowhy/causal_estimators/regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
/Users/hc.cho/Workspace/decision-layer/.venv/lib/python3.11/site-packages/dowhy/causal_estimators/regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is d

,field,value
0,value,2.023501
1,estimator,backdoor.linear_regression
2,contrast,"[0, 1]"
3,target,ate
4,n_units,2000
5,graph_edges,"[[treatment, outcome], [adjustment_0, treatmen..."
6,estimand,Estimand type: EstimandType.NONPARAMETRIC_ATE\...
7,placebo_difference,0.002933
8,seed,7
9,support,"{'min': 0.25189573942649124, 'max': 0.74455237..."


### 일반 Python에서도 사용

같은 계산 함수에 DataFrame을 직접 전달합니다. Method 실행과 같은 값이 나오는지 확인합니다.

In [214]:
frame = example_data.binary_regression_frame()
standalone_result = estimate_binary_regression(frame, ["baseline"])

assert abs(standalone_result["value"] - 2) < 0.15
assert standalone_result["value"] == dowhy_trial.result.primary.data["value"]
show_table(standalone_result)


/Users/hc.cho/Workspace/decision-layer/.venv/lib/python3.11/site-packages/dowhy/causal_estimators/regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
/Users/hc.cho/Workspace/decision-layer/.venv/lib/python3.11/site-packages/dowhy/causal_estimators/regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
/Users/hc.cho/Workspace/decision-layer/.venv/lib/python3.11/site-packages/dowhy/causal_estimators/regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is d

,field,value
0,value,2.023501
1,estimator,backdoor.linear_regression
2,contrast,"[0, 1]"
3,target,ate
4,n_units,2000
5,graph_edges,"[(treatment, outcome), (adjustment_0, treatmen..."
6,estimand,Estimand type: EstimandType.NONPARAMETRIC_ATE\...
7,placebo_difference,0.002933
8,seed,7
9,support,"{'min': 0.25189573942649124, 'max': 0.74455237..."


### 실패 확인

분석 단위가 중복된 연습 데이터를 넣습니다. Method는 추정을 거절하고 이유를 반환합니다.

In [215]:
refusal_session = await example_data.duplicate_unit_session()
refusal_session.register(my_binary_regression)

refused_trial = await refusal_session.run(
    my_binary_regression,
    outcome="cube://practice/observations/value",
    treatment="cube://practice/observations/treated",
    confounders=["cube://practice/observations/baseline"],
    unit="cube://practice/observations/id",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
)

assert refused_trial.result.status == "refused"
show_table([
    {"status": refused_trial.result.status, "reason": warning}
    for warning in refused_trial.result.warnings
])
refusal_session.close()


,status,reason
0,refused,Entity observations need unique non-null analy...


### 기본 제공 DoWhy 분석 두 가지 실행하기

- `stats.binary_matching`: 처치 전 조건이 가까운 비교 개체를 매칭해 처치 집단의 차이(ATT)를 추정합니다.
- `stats.regression_adjustment`: DoWhy OLS로 조건을 조정해 처치 1단위 증가당 변화량을 추정합니다. 특정 두 값의 대비도 선택할 수 있습니다.

아래 코드는 같은 입력 네 개를 받는 등록 Method를 실행합니다. 합성 데이터의 알려진 차이는 2입니다. 실제 인과 가정은 검증되지 않았고 신뢰구간은 제공하지 않습니다. 전체 조회와 실행 코드는 `dowhy_basics.py`에서 확인할 수 있습니다.


In [ ]:
from examples.methods.statistics.dowhy_basics import demonstrate

show_table(await demonstrate())


### 연속형 처치: 변수만 넣어 실행하기

할인율은 0.05에서 0.30 사이이고 구매액은 `100 + 50 × 할인율 + 3 × 이전 특성`인 합성 데이터입니다.

- 기본 호출은 할인율 1.0 증가당 조정된 변화량 **50**을 반환합니다. 1%p(0.01)당 변화량은 **0.5**에 해당합니다.
- 10%에서 20%로 바뀌는 특정 대비를 지정하면 **5**를 반환합니다.
- 처치의 단위는 semantic 정의에서 확인합니다. Method는 퍼센트 단위를 임의로 바꾸지 않습니다.

1단위 결과는 선형 모형의 일정한 기울기입니다. 관측 범위 밖의 실제 처치 변경에도 같은 효과가 난다는 근거는 아닙니다.


In [ ]:
from examples.methods.statistics import continuous_regression


In [ ]:
continuous_session = await continuous_regression.example_session()

continuous_trial = await continuous_session.run(
    "stats.regression_adjustment",
    outcome="cube://practice/observations/value",
    treatment="cube://practice/observations/rate",
    confounders=["cube://practice/observations/baseline"],
    unit="cube://practice/observations/id",
    scope=Scope(("2026-02-01", "2026-02-28"), "cube://practice/observations/date"),
)
show_table([{
    "result": "per stored treatment unit",
    "value": continuous_trial.result.primary.data["value"],
}])
continuous_session.close()


In [ ]:
continuous_session = await continuous_regression.example_session()

contrast_trial = await continuous_session.run(
    "stats.regression_adjustment",
    outcome="cube://practice/observations/value",
    treatment="cube://practice/observations/rate",
    confounders=["cube://practice/observations/baseline"],
    unit="cube://practice/observations/id",
    scope=Scope(("2026-02-01", "2026-02-28"), "cube://practice/observations/date"),
    control_value=0.10,
    treatment_value=0.20,
)
show_table([{
    "result": "0.10 to 0.20",
    "value": contrast_trial.result.primary.data["value"],
}])
continuous_session.close()


## 6. 실제 Cube 데이터로 실행하기

이제 준비된 응답 대신 연결한 Cube에서 조회합니다. 연결 셀의 `RUN_LIVE`를 `True`로 바꾸고 실행합니다. 기본 연결 값은 로컬 Journey 샘플용입니다.

이 절도 로컬 Python 실행입니다. 서버 등록이나 웹 목록 변경은 하지 않습니다.


In [216]:
import os
from decision_layer.semantic.credentials import ServiceCredentials

RUN_LIVE = False
live = None

if RUN_LIVE:
    live = await MethodSession.cube(
        os.environ.get("DL_JOURNEY_CUBE_URL", "http://127.0.0.1:4000/cubejs-api/v1"),
        ServiceCredentials(
            os.environ.get(
                "DL_JOURNEY_CUBE_SECRET",
                "local-example-secret-change-me-0123456789",
            ),
            (),
        ),
        instance="journey",
    )

show_table([{
    "status": "connected" if live is not None else "skipped",
    "reason": "Cube connected" if live is not None else "Set RUN_LIVE = True in this connection cell",
}])

,status,reason
0,skipped,Set RUN_LIVE = True in this connection cell


### 연결된 지표와 분류 기준 찾기

실제 catalog에서 `receipts`와 `department`의 경로를 확인합니다. 아래 실행 코드에 이 경로를 넣습니다.


In [217]:
if live is not None:
    objects = [
        *live.catalog.search(kind="measure", text="receipts"),
        *live.catalog.search(kind="dimension", text="department"),
    ]
    show_table([
        {"title": obj.title, "kind": obj.kind, "path": obj.ref}
        for obj in objects
    ])
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])

,status,reason
0,skipped,Enable the Cube connection above


### 그룹별 조회

판매액을 department별로 조회합니다. `limit`이나 기간을 바꿔보세요.

In [218]:
if live is not None:
    live.register(my_by_group, replace=True)
    live_group_trial = await live.run(
        my_by_group,
        selected_measures=["cube://journey/transaction/receipts"],
        group_by=["cube://journey/product/department"],
        scope=Scope(
            ("2001-01-01", "2001-06-30"),
            "cube://journey/transaction/analysis_date",
        ),
        limit=3,
    )
    show_table(live_group_trial.result.primary.data)
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])

,status,reason
0,skipped,Enable the Cube connection above


### 일반 계산 함수도 같은 Cube 연결 사용

위에서 확인한 Journey 참조를 사용해 같은 `native_rank`를 실행합니다. 소스 집계 후 전체 그룹을 계산 함수에 전달하며, 허용 행 수를 넘으면 거부합니다. 이 실행은 수취액 순위이며 인과 분석이 아닙니다.


In [ ]:
if live is not None:
    live.register(native_rank, replace=True)
    live_native_trial = await live.run(
        native_rank,
        metric="cube://journey/transaction/receipts",
        group="cube://journey/product/department",
        scope=Scope(("2001-01-01", "2001-06-30"), "cube://journey/transaction/analysis_date"),
    )
    show_table(live_native_trial.result.primary.data if live_native_trial.result.primary else [])
    show_table([item.model_dump() for item in live_native_trial.result.validation])
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])


### 집단 비교

GROCERY의 store 364를 동종 집단과 전체 집단과 비교합니다. 실제 존재하는 다른 store나 department로 바꿔보세요.

In [219]:
if live is not None:
    live.register(my_peer, replace=True)
    live_peer_trial = await live.run(
        my_peer,
        measure="cube://journey/transaction/coupon_line_rate",
        subject="cube://journey/transaction/store",
        category="cube://journey/product/department",
        scope=Scope(
            ("2001-01-01", "2001-06-30"),
            "cube://journey/transaction/analysis_date",
        ),
        subject_value="364",
        category_value="GROCERY",
    )
    show_table(live_peer_trial.result.primary.data)
    show_table([
        {"query": index, "label": attempt.label, "status": attempt.status}
        for index, attempt in enumerate(live_peer_trial.attempts, start=1)
    ])
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])

,status,reason
0,skipped,Enable the Cube connection above


### 마무리

로컬 연결을 닫습니다. 만든 Method를 웹이나 MCP에서 제공하려면 함수를 Python 파일에 저장하고, 서버 시작 시 등록한 뒤 배포합니다.


In [220]:
if live is not None:
    live.close()
    live = None

show_table([{"status": "closed"}])

,status
0,closed
